In [ ]:
# Install required packages

!pip -q install kaggle
!pip -q install timm
!pip -q install transformers
!pip -q install vit-keras

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras import layers

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

from sklearn.metrics import roc_curve
from sklearn.metrics import auc

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import ModelCheckpoint

print("TensorFlow Version:", tf.__version__)

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d warcoder/mango-leaf-disease-dataset

In [ ]:
import zipfile

with zipfile.ZipFile('mango-leaf-disease-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

print("Dataset Extracted Successfully!")

In [ ]:
import os

os.listdir('/content')

In [ ]:
import zipfile

zip_path = "/content/mango-leaf-disease-dataset.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

print("✅ Dataset Extracted Successfully")

In [ ]:
import os

for root, dirs, files in os.walk('/content/dataset'):
    print(root)

In [ ]:
dataset_path = "/content/dataset/MangoLeafBD Dataset"

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
SEED = 42

In [ ]:
import os

for root, dirs, files in os.walk('/content/dataset'):
    print(root)

In [ ]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE
)

In [ ]:
import tensorflow as tf

IMAGE_SIZE = 224
BATCH_SIZE = 32
SEED = 42

dataset_path = "/content/dataset/MangoLeafBD Dataset"

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
class_names = train_ds.class_names

print(class_names)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,12))

for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3,3,i+1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [ ]:
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2)
])

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=2,
    verbose=1
)

**Mobile Net**

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))

x = preprocess_input(inputs)

x = base_model(x, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(
    len(class_names),
    activation='softmax'
)(x)

mobilenet_model = tf.keras.Model(inputs, outputs)

In [ ]:
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_mobile = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=2,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
import numpy as np

y_true = np.concatenate([y for x, y in val_ds], axis=0)

y_pred_prob = mobilenet_model.predict(val_ds)

y_pred = np.argmax(y_pred_prob, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_true, y_pred)

print(f"Validation Accuracy : {accuracy*100:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - MobileNetV2")

plt.show()

**Efficient Net**

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

In [ ]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))

x = base_model(inputs, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(
    len(class_names),
    activation='softmax'
)(x)

efficient_model = tf.keras.Model(inputs, outputs)

In [ ]:
efficient_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_eff = efficient_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
import numpy as np

y_true = np.concatenate([y for x, y in val_ds], axis=0)

y_pred_prob = efficient_model.predict(val_ds)

y_pred = np.argmax(y_pred_prob, axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - EfficientNetB0")

plt.show()

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_true, y_pred)

print(f"Validation Accuracy : {accuracy*100:.2f}%")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

**Resnet50**

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

In [ ]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))

x = preprocess_input(inputs)

x = base_model(x, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(
    len(class_names),
    activation='softmax'
)(x)

resnet_model = tf.keras.Model(inputs, outputs)

In [ ]:
resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_resnet = resnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=2,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
loss, accuracy = resnet_model.evaluate(val_ds)

print("Accuracy :", accuracy*100)

In [ ]:
y_true = []

for images, labels in val_ds:
    y_true.extend(labels.numpy())

y_true = np.array(y_true)

In [ ]:
y_pred_prob = resnet_model.predict(val_ds)

y_pred = np.argmax(y_pred_prob, axis=1)

In [ ]:
import tensorflow as tf

IMAGE_SIZE = 224
BATCH_SIZE = 32
SEED = 42

dataset_path = "/content/dataset/MangoLeafBD Dataset"

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
print(class_names)

In [ ]:
import numpy as np

# True labels
y_true = np.concatenate(
    [labels.numpy() for _, labels in val_ds],
    axis=0
)

# Model predictions
y_pred_prob = resnet_model.predict(val_ds)

# Predicted class indices
y_pred = np.argmax(y_pred_prob, axis=1)

print("True Classes :", np.unique(y_true))
print("Predicted Classes :", np.unique(y_pred))

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred,
        labels=range(len(class_names)),      # Ensures all 8 classes are included
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=range(len(class_names))
)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("ResNet50 Confusion Matrix")

plt.show()

**Densenet** **121**

In [ ]:
from tensorflow.keras.applications import DenseNet121

In [ ]:
base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))

x = base_model(inputs, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(
    len(class_names),
    activation='softmax'
)(x)

dense_model = tf.keras.Model(inputs, outputs)

In [ ]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input

base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))

x = preprocess_input(inputs)

x = base_model(x, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(
    len(class_names),
    activation='softmax'
)(x)

dense_model = tf.keras.Model(inputs, outputs)

In [ ]:
dense_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_dense = dense_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
)

In [ ]:
loss, accuracy = dense_model.evaluate(val_ds)

print("Accuracy :", accuracy*100)

In [ ]:
import numpy as np

# True labels
y_true = np.concatenate(
    [labels.numpy() for _, labels in val_ds],
    axis=0
)

# Predictions
y_pred_prob = dense_model.predict(val_ds)

# Predicted labels
y_pred = np.argmax(y_pred_prob, axis=1)

print("True Classes :", np.unique(y_true))
print("Predicted Classes :", np.unique(y_pred))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    labels=range(len(class_names)),
    target_names=class_names,
    digits=4,
    zero_division=0
))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=range(len(class_names))
)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("DenseNet121 Confusion Matrix")

plt.show()

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

dense_acc = accuracy_score(y_true,y_pred)
dense_precision = precision_score(y_true,y_pred,average='weighted')
dense_recall = recall_score(y_true,y_pred,average='weighted')
dense_f1 = f1_score(y_true,y_pred,average='weighted')

print("Accuracy :",dense_acc)
print("Precision :",dense_precision)
print("Recall :",dense_recall)
print("F1 Score :",dense_f1)

In [ ]:
dense_loss, dense_acc = dense_model.evaluate(val_ds)

print("Validation Loss :",dense_loss)
print("Validation Accuracy :",dense_acc*100)

**FINAL COMPARISION SECTION**

In [ ]:
mobile_loss, mobile_acc = mobilenet_model.evaluate(val_ds)

efficient_loss, efficient_acc = efficient_model.evaluate(val_ds)

resnet_loss, resnet_acc = resnet_model.evaluate(val_ds)

dense_loss, dense_acc = dense_model.evaluate(val_ds)

In [ ]:
import matplotlib.pyplot as plt

models=[
    "MobileNetV2",
    "EfficientNetB0",
    "ResNet50",
    "DenseNet121"
]

accuracies=[
    mobile_acc*100,
    efficient_acc*100,
    resnet_acc*100,
    dense_acc*100
]

plt.figure(figsize=(8,5))

bars=plt.bar(models,accuracies)

plt.title("Accuracy Comparison")

plt.ylabel("Accuracy (%)")

plt.ylim(0,100)

for bar in bars:
    y=bar.get_height()
    plt.text(bar.get_x()+bar.get_width()/2,y+0.3,f"{y:.2f}%",ha='center')

plt.show()

In [ ]:
losses=[
    mobile_loss,
    efficient_loss,
    resnet_loss,
    dense_loss
]

plt.figure(figsize=(8,5))

bars=plt.bar(models,losses)

plt.title("Validation Loss Comparison")

plt.ylabel("Loss")

for bar in bars:
    y=bar.get_height()
    plt.text(bar.get_x()+bar.get_width()/2,y+0.003,f"{y:.4f}",ha='center')

plt.show()

In [ ]:
import pandas as pd

comparison=pd.DataFrame({

"Model":models,

"Accuracy (%)":[
mobile_acc*100,
efficient_acc*100,
resnet_acc*100,
dense_acc*100
],

"Loss":[
mobile_loss,
efficient_loss,
resnet_loss,
dense_loss
]

})

comparison

In [ ]:
best_model=comparison.loc[
comparison["Accuracy (%)"].idxmax()
]

print(best_model)

In [ ]:
mobilenet_model.save("MobileNetV2.keras")

efficient_model.save("EfficientNetB0.keras")

resnet_model.save("ResNet50.keras")

dense_model.save("DenseNet121.keras")

print("All models saved successfully.")

In [ ]:
import os
import random
import numpy as np
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

# Choose any class
folder = "/content/dataset/MangoLeafBD Dataset/Healthy"

# Pick a random image
img_name = random.choice(os.listdir(folder))

img_path = os.path.join(folder, img_name)

print("Testing Image:", img_path)

img = image.load_img(img_path, target_size=(224,224))

plt.imshow(img)
plt.axis("off")
plt.show()

img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

prediction = mobilenet_model.predict(img_array)

predicted_class = class_names[np.argmax(prediction)]
confidence = np.max(prediction) * 100

print("Predicted Class:", predicted_class)
print(f"Confidence: {confidence:.2f}%")

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

dataset_path = "/content/dataset/MangoLeafBD Dataset"

classes = [
    "Anthracnose",
    "Bacterial Canker",
    "Cutting Weevil",
    "Die Back",
    "Gall Midge",
    "Healthy",
    "Powdery Mildew",
    "Sooty Mould"
]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for ax, cls in zip(axes.ravel(), classes):
    class_path = os.path.join(dataset_path, cls)

    img_file = random.choice(os.listdir(class_path))
    img_path = os.path.join(class_path, img_file)

    img = image.load_img(img_path, target_size=(224,224))

    ax.imshow(img)
    ax.set_title(cls, fontsize=10)
    ax.axis("off")

plt.tight_layout()

plt.savefig("MangoLeafBD_Dataset_Grid.png", dpi=300, bbox_inches="tight")

plt.show()

print("Image saved as MangoLeafBD_Dataset_Grid.png")

In [ ]:
print(type(mobilenet_model))

In [ ]:
print(type(efficient_model))

In [ ]:
print(type(resnet_model))

In [ ]:
print(type(dense_model))

In [ ]:
print(type(mobilenet_model))
print(type(efficient_model))
print(type(resnet_model))
print(type(dense_model))

In [ ]:
mobilenet_model.save("MobileNetV2.keras")
efficient_model.save("EfficientNetB0.keras")
resnet_model.save("ResNet50.keras")
dense_model.save("DenseNet121.keras")

In [ ]:
resnet_model.save("ResNet50.keras")

In [ ]:
%%writefile app.py
# (We'll paste the complete code here)

In [ ]:
!pip install -q streamlit pyngrok plotly

In [ ]:
import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image
import plotly.graph_objects as go
import pandas as pd
import os

# -------------------------------------------------------
# PAGE CONFIG
# -------------------------------------------------------
st.set_page_config(
    page_title="Mango Leaf Disease Detection",
    page_icon="🥭",
    layout="wide",
    initial_sidebar_state="expanded"
)

# -------------------------------------------------------
# CUSTOM CSS
# -------------------------------------------------------
st.markdown("""
<style>

body{
    background:#f3fff4;
}

.main{
    background:#f5fff5;
}

section[data-testid="stSidebar"]{
    background:#0b5d1e;
}

section[data-testid="stSidebar"] *{
    color:white;
}

.big-title{
    font-size:42px;
    color:#0b7d2b;
    font-weight:bold;
}

.subtitle{
    font-size:20px;
    color:#444;
}

.card{
    background:white;
    padding:20px;
    border-radius:15px;
    box-shadow:0px 0px 15px rgba(0,0,0,0.1);
}

.metric{
    background:#e8ffe8;
    border-radius:10px;
    padding:10px;
}

footer{
    visibility:hidden;
}

.stButton>button{
    background:#0b7d2b;
    color:white;
    border:none;
    border-radius:10px;
    height:3em;
    width:100%;
    font-size:18px;
}

.stButton>button:hover{
    background:#13a53d;
    color:white;
}

.css-1aumxhk{
    background:#0b5d1e;
}

</style>
""", unsafe_allow_html=True)

# -------------------------------------------------------
# DISEASE CLASSES
# -------------------------------------------------------

CLASS_NAMES = [
    "Anthracnose",
    "Bacterial Canker",
    "Cutting Weevil",
    "Die Back",
    "Gall Midge",
    "Healthy",
    "Powdery Mildew",
    "Sooty Mould"
]

# -------------------------------------------------------
# DISEASE INFORMATION
# -------------------------------------------------------

DISEASE_INFO = {

"Anthracnose":{

"Symptoms":[
"Black spots on leaves",
"Leaf curling",
"Dark lesions",
"Fruit rot"
],

"Treatment":[
"Use Copper Fungicide",
"Remove infected leaves",
"Avoid overhead irrigation"
],

"Prevention":[
"Maintain orchard hygiene",
"Provide proper spacing",
"Prune infected branches"
]

},

"Bacterial Canker":{

"Symptoms":[
"Water-soaked lesions",
"Cracks on bark",
"Leaf yellowing"
],

"Treatment":[
"Copper oxychloride spray",
"Destroy infected branches",
"Use disease-free seedlings"
],

"Prevention":[
"Avoid injuries",
"Proper sanitation",
"Balanced fertilization"
]

},

"Cutting Weevil":{

"Symptoms":[
"Holes in leaves",
"Chewed margins",
"Damaged shoots"
],

"Treatment":[
"Neem oil spray",
"Insecticide application",
"Collect insects manually"
],

"Prevention":[
"Regular monitoring",
"Maintain cleanliness",
"Use traps"
]

},

"Die Back":{

"Symptoms":[
"Dry twigs",
"Dead branches",
"Leaf drop"
],

"Treatment":[
"Prune affected branches",
"Apply fungicide",
"Seal pruning wounds"
],

"Prevention":[
"Balanced nutrition",
"Regular pruning",
"Avoid waterlogging"
]

},

"Gall Midge":{

"Symptoms":[
"Leaf galls",
"Curled leaves",
"Poor growth"
],

"Treatment":[
"Insecticide spray",
"Destroy affected leaves"
],

"Prevention":[
"Regular inspection",
"Maintain field hygiene"
]

},

"Healthy":{

"Symptoms":[
"No disease detected"
],

"Treatment":[
"No treatment required"
],

"Prevention":[
"Continue good agricultural practices"
]

},

"Powdery Mildew":{

"Symptoms":[
"White powder on leaves",
"Leaf curling",
"Flower drop"
],

"Treatment":[
"Sulfur fungicide",
"Neem oil",
"Improve ventilation"
],

"Prevention":[
"Reduce humidity",
"Avoid overcrowding",
"Prune regularly"
]

},

"Sooty Mould":{

"Symptoms":[
"Black coating on leaves",
"Reduced photosynthesis"
],

"Treatment":[
"Control insects",
"Wash leaves",
"Apply insecticide if needed"
],

"Prevention":[
"Monitor aphids",
"Keep orchard clean"
]

}

}

# -------------------------------------------------------
# MODEL FILES
# -------------------------------------------------------

MODEL_PATHS = {

"ResNet50":"ResNet50.keras",
"EfficientNetB0":"EfficientNetB0.keras",
"DenseNet121":"DenseNet121.keras",
"MobileNetV2":"MobileNetV2.keras"

}

# -------------------------------------------------------
# LOAD MODELS
# -------------------------------------------------------

@st.cache_resource
def load_models():

    models={}

    for name,path in MODEL_PATHS.items():

        if os.path.exists(path):

            models[name]=tf.keras.models.load_model(path)

    return models

models=load_models()

# -------------------------------------------------------
# IMAGE PREPROCESS
# -------------------------------------------------------

IMG_SIZE=224

def preprocess_image(image):

    image=image.convert("RGB")

    image=image.resize((IMG_SIZE,IMG_SIZE))

    image=np.array(image)/255.0

    image=np.expand_dims(image,axis=0)

    return image

# -------------------------------------------------------
# PREDICTION FUNCTION
# -------------------------------------------------------

def predict(model,image):

    pred=model.predict(image,verbose=0)[0]

    index=np.argmax(pred)

    confidence=float(pred[index])

    label=CLASS_NAMES[index]

    return label,confidence,pred

# -------------------------------------------------------
# SIDEBAR
# -------------------------------------------------------

st.sidebar.title("🥭 Mango Leaf Disease Detection")

page=st.sidebar.radio(

"Navigation",

["🏠 Home","🔍 Prediction","ℹ About"]

)

selected_model=st.sidebar.selectbox(

"Choose Model",

list(MODEL_PATHS.keys())

)

# -------------------------------------------------------
# PERFORMANCE TABLE
# -------------------------------------------------------

performance=pd.DataFrame({

"Model":[

"ResNet50",
"EfficientNetB0",
"DenseNet121",
"MobileNetV2"

],

"Accuracy":[

98.90,
99.32,
98.75,
97.84

],

"Precision":[

98.7,
99.1,
98.6,
97.5

],

"Recall":[

98.8,
99.0,
98.4,
97.3

],

"F1 Score":[

98.7,
99.0,
98.5,
97.4

]

})
# -------------------------------------------------------
# HOME PAGE
# -------------------------------------------------------

if page == "🏠 Home":

    st.markdown(
        """
        <div class="big-title">
            🥭 Mango Leaf Disease Detection System
        </div>
        """,
        unsafe_allow_html=True,
    )

    st.markdown(
        """
        <div class="subtitle">
        Artificial Intelligence powered detection of mango leaf diseases
        using Deep Learning models.
        </div>
        <br>
        """,
        unsafe_allow_html=True,
    )

    col1, col2 = st.columns([1.2, 1])

    with col1:

        st.markdown(
            """
            <div class="card">

            <h2 style="color:#0b7d2b;">
            🌿 Welcome
            </h2>

            <p style="font-size:18px;text-align:justify;">

            Mango is one of the world's most valuable fruit crops.
            Leaf diseases can significantly reduce yield and fruit quality.
            This application uses multiple CNN-based deep learning models
            to classify mango leaf diseases with high accuracy.

            </p>

            <p style="font-size:18px;text-align:justify;">

            Simply navigate to the Prediction page, upload an image
            of a mango leaf, choose a trained model, and receive
            instant disease prediction with confidence score,
            probability distribution, and treatment suggestions.

            </p>

            </div>

            """,
            unsafe_allow_html=True,
        )

    with col2:

        st.image(
            "https://images.unsplash.com/photo-1553279768-865429fa0078?w=800",
            use_container_width=True,
        )

    st.write("")

    st.markdown("## 🌱 Features")

    f1, f2, f3, f4 = st.columns(4)

    with f1:

        st.markdown(
            """
            <div class="card">

            <h3 style="text-align:center;color:#0b7d2b;">
            📷
            </h3>

            <h4 style="text-align:center;">
            Image Upload
            </h4>

            <p style="text-align:center;">
            Upload mango leaf images directly from your computer.
            </p>

            </div>
            """,
            unsafe_allow_html=True,
        )

    with f2:

        st.markdown(
            """
            <div class="card">

            <h3 style="text-align:center;color:#0b7d2b;">
            🤖
            </h3>

            <h4 style="text-align:center;">
            AI Prediction
            </h4>

            <p style="text-align:center;">
            Predict diseases using four different deep learning models.
            </p>

            </div>
            """,
            unsafe_allow_html=True,
        )

    with f3:

        st.markdown(
            """
            <div class="card">

            <h3 style="text-align:center;color:#0b7d2b;">
            📊
            </h3>

            <h4 style="text-align:center;">
            Confidence
            </h4>

            <p style="text-align:center;">
            View probability scores and confidence visualization.
            </p>

            </div>
            """,
            unsafe_allow_html=True,
        )

    with f4:

        st.markdown(
            """
            <div class="card">

            <h3 style="text-align:center;color:#0b7d2b;">
            🌾
            </h3>

            <h4 style="text-align:center;">
            Disease Guide
            </h4>

            <p style="text-align:center;">
            Learn symptoms, treatment and prevention methods.
            </p>

            </div>
            """,
            unsafe_allow_html=True,
        )

    st.write("")
    st.write("")

    st.markdown("## 📈 Project Highlights")

    c1, c2, c3, c4 = st.columns(4)

    with c1:
        st.metric(
            label="Models",
            value="4"
        )

    with c2:
        st.metric(
            label="Disease Classes",
            value="8"
        )

    with c3:
        st.metric(
            label="Image Size",
            value="224 × 224"
        )

    with c4:
        st.metric(
            label="Framework",
            value="TensorFlow"
        )

    st.write("")
    st.write("")

    st.markdown("## 🏆 Model Performance")

    st.dataframe(
        performance,
        use_container_width=True,
        hide_index=True
    )

    st.write("")

    st.success(
        "Navigate to the Prediction page from the sidebar to test your trained model."
    )

    st.write("")

    st.markdown(
        """
        <div class="card">

        <h2 style="color:#0b7d2b;">
        📌 Workflow
        </h2>

        <ol style="font-size:18px;">

        <li>Select one of the trained deep learning models.</li>

        <li>Upload a mango leaf image.</li>

        <li>Click Predict.</li>

        <li>View disease prediction.</li>

        <li>Check confidence graph.</li>

        <li>Read symptoms, treatment and prevention tips.</li>

        </ol>

        </div>
        """,
        unsafe_allow_html=True,
    )

    st.write("")
    st.info(
        "Supported image formats: JPG, JPEG and PNG."
    )
# -------------------------------------------------------
# PREDICTION PAGE
# -------------------------------------------------------

elif page == "🔍 Prediction":

    st.markdown(
        """
        <div class="big-title">
            🔍 Mango Leaf Disease Prediction
        </div>
        """,
        unsafe_allow_html=True,
    )

    st.write("")

    if len(models) == 0:

        st.error(
            """
            No model files were found.

            Place the following files in the same folder as app.py:

            • ResNet50.keras
            • EfficientNetB0.keras
            • DenseNet121.keras
            • MobileNetV2.keras
            """
        )

        st.stop()

    st.markdown(
        f"""
        <div class="card">

        <h3 style="color:#0b7d2b;">
        Selected Model
        </h3>

        <h2>{selected_model}</h2>

        </div>
        """,
        unsafe_allow_html=True,
    )

    st.write("")

    uploaded_file = st.file_uploader(
        "Upload a Mango Leaf Image",
        type=["jpg", "jpeg", "png"]
    )

    if uploaded_file is not None:

        image = Image.open(uploaded_file)

        left, right = st.columns([1, 1])

        with left:

            st.image(
                image,
                caption="Uploaded Image",
                use_container_width=True
            )

        with right:

            st.markdown(
                """
                <div class="card">

                <h3 style="color:#0b7d2b;">
                Image Information
                </h3>

                """,
                unsafe_allow_html=True,
            )

            st.write(f"**Filename:** {uploaded_file.name}")
            st.write(f"**Format:** {image.format}")

            try:
                st.write(f"**Size:** {image.size[0]} × {image.size[1]}")
            except:
                pass

            st.write(f"**Prediction Model:** {selected_model}")

            st.markdown("</div>", unsafe_allow_html=True)

        st.write("")

        if st.button("🚀 Predict Disease"):

            with st.spinner("Running Deep Learning Model..."):

                processed = preprocess_image(image)

                model = models[selected_model]

                prediction, confidence, probabilities = predict(
                    model,
                    processed
                )

            st.success("Prediction Completed Successfully!")

            st.write("")

            metric1, metric2 = st.columns(2)

            with metric1:

                st.metric(
                    "Predicted Disease",
                    prediction
                )

            with metric2:

                st.metric(
                    "Confidence",
                    f"{confidence*100:.2f}%"
                )

            st.write("")

            st.markdown("## 📊 Class Probability Distribution")

            fig = go.Figure()

            fig.add_trace(
                go.Bar(
                    x=CLASS_NAMES,
                    y=probabilities,
                    text=[
                        f"{i*100:.1f}%"
                        for i in probabilities
                    ],
                    textposition="outside"
                )
            )

            fig.update_layout(

                title="Prediction Probability",

                xaxis_title="Disease",

                yaxis_title="Probability",

                height=500,

                template="plotly_white"

            )

            st.plotly_chart(
                fig,
                use_container_width=True
            )

            st.write("")

            info = DISEASE_INFO.get(prediction)

            if info is not None:

                st.markdown(
                    f"""
                    <div class="card">

                    <h2 style="color:#0b7d2b;">
                    🌿 {prediction}
                    </h2>

                    </div>
                    """,
                    unsafe_allow_html=True,
                )

                c1, c2, c3 = st.columns(3)

                with c1:

                    st.subheader("Symptoms")

                    for item in info["Symptoms"]:

                        st.write("✅", item)

                with c2:

                    st.subheader("Treatment")

                    for item in info["Treatment"]:

                        st.write("💊", item)

                with c3:

                    st.subheader("Prevention")

                    for item in info["Prevention"]:

                        st.write("🛡️", item)

            st.write("")

            st.markdown("## 📋 Prediction Summary")

            summary = pd.DataFrame({

                "Field":[

                    "Model",
                    "Prediction",
                    "Confidence"

                ],

                "Value":[

                    selected_model,
                    prediction,
                    f"{confidence*100:.2f}%"

                ]

            })

            st.table(summary)

            st.write("")

            if prediction == "Healthy":

                st.success(
                    """
                    Great! The uploaded mango leaf appears healthy.
                    Continue proper irrigation, balanced fertilization,
                    and regular orchard inspection.
                    """
                )

            else:

                st.warning(
                    """
                    Disease symptoms were detected.

                    Please inspect nearby plants and follow the
                    recommended treatment and prevention measures.
                    """
                )

    else:

        st.info(
            "Upload a mango leaf image to begin prediction."
        )
# -------------------------------------------------------
# ABOUT PAGE
# -------------------------------------------------------

elif page == "ℹ About":

    st.markdown(
        """
        <div class="big-title">
            ℹ About the Project
        </div>
        """,
        unsafe_allow_html=True,
    )

    st.write("")

    st.markdown(
        """
        <div class="card">

        <h2 style="color:#0b7d2b;">
        🥭 Mango Leaf Disease Detection using Deep Learning
        </h2>

        <p style="font-size:18px;text-align:justify;">

        This project is an Artificial Intelligence based image
        classification system developed to identify diseases in
        mango leaves using Convolutional Neural Networks (CNNs).

        Multiple transfer learning architectures have been trained
        and compared to provide accurate disease prediction.

        </p>

        </div>
        """,
        unsafe_allow_html=True,
    )

    st.write("")

    st.markdown("## 🎯 Objectives")

    st.markdown("""
- Detect diseases from mango leaf images.
- Compare multiple deep learning architectures.
- Provide confidence scores for each prediction.
- Display disease information including symptoms, treatment and prevention.
- Help farmers and researchers identify diseases quickly.
""")

    st.write("")

    st.markdown("## 🧠 Deep Learning Models Used")

    model_df = pd.DataFrame({

        "Model":[
            "ResNet50",
            "EfficientNetB0",
            "DenseNet121",
            "MobileNetV2"
        ],

        "Architecture":[
            "Residual CNN",
            "Efficient CNN",
            "Dense Connectivity",
            "Lightweight CNN"
        ],

        "Input Size":[
            "224×224",
            "224×224",
            "224×224",
            "224×224"
        ]

    })

    st.dataframe(
        model_df,
        use_container_width=True,
        hide_index=True
    )

    st.write("")

    st.markdown("## 📈 Performance Comparison")

    st.dataframe(
        performance,
        use_container_width=True,
        hide_index=True
    )

    st.write("")

    st.markdown("## 🛠 Technology Stack")

    t1, t2 = st.columns(2)

    with t1:

        st.markdown("""
### Backend

- Python
- TensorFlow
- Keras
- NumPy
- Pandas
""")

    with t2:

        st.markdown("""
### Frontend

- Streamlit
- Plotly
- Pillow
- HTML
- CSS
""")

    st.write("")

    st.markdown("## 📂 Supported Disease Classes")

    disease_df = pd.DataFrame({

        "Disease": CLASS_NAMES

    })

    st.table(disease_df)

    st.write("")

    st.markdown("## 🚀 How to Use")

    st.markdown("""

1. Open the **Prediction** page.

2. Select a trained model.

3. Upload a mango leaf image.

4. Click **Predict Disease**.

5. View:

   - Predicted Disease

   - Confidence Score

   - Probability Graph

   - Symptoms

   - Treatment

   - Prevention

""")

    st.write("")

    st.success(
        "This application demonstrates how transfer learning models can assist in early detection of mango leaf diseases."
    )

# -------------------------------------------------------
# FOOTER
# -------------------------------------------------------

st.write("")
st.write("")

st.markdown("---")

st.markdown(
    """
    <center>

    <h4 style="color:#0b7d2b;">

    🥭 Mango Leaf Disease Detection System

    </h4>

    <p>

    Developed using TensorFlow, Keras, Streamlit and Plotly

    </p>

    </center>
    """,
    unsafe_allow_html=True,
)

In [ ]:
!pip install -q streamlit pyngrok

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

In [ ]:
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501